# Example: No-Reset Benchmark

This notebook demonstrates the no-reset benchmarking API with several online CPD algorithms. It follows the workflow introduced in the core API notebooks: create a dataset, configure algorithms, build benchmark entries, run the benchmark, and inspect cached benchmark traces.

The `extreme_mean_shifts` preset keeps the comparison readable while still producing clear transitions. Most configured algorithms detect mean shifts well, while **Variance CUSUM** is designed for scale changes and will struggle, making it a useful case study.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

from pysatl_cpd.algorithms.online import (
    CrosierCusum,
    PageTwoSidedCusum,
    ShewhartControlChart,
    VarianceTwoSidedCUSUM,
)
from pysatl_cpd.algorithms.online.bayesian import UnivariateGaussianConjugateBOCPD
from pysatl_cpd.analysis.visualization import (
    DrawBackend,
    MultivariateTimeseriesVisualizer,
    OnlineCpdPlotter,
    VerticalLineVisualComponent,
)
from pysatl_cpd.analysis.visualization.benchmarking import (
    ARLBasedMetricVisualizer,
    BenchmarkPlotter,
    PrAucVisualizer,
)
from pysatl_cpd.benchmark.online.noreset import (
    NoResetPolicyKind,
    OnlineNoResetBenchmark,
    OnlineNoResetBenchmarkEntry,
)
from pysatl_cpd.benchmark.online.noreset.thresholds.ranges import AutoThresholdsRange
from pysatl_cpd.benchmark.online.noreset.tooling.bisegment_cut import BisegmentCut
from pysatl_cpd.benchmark.registry import BenchmarkRegistry
from pysatl_cpd.data.generator import preset_dataset
from pysatl_cpd.data.providers.transformers import ColumnsSelectorTransformer

## Dataset

We use the `extreme_mean_shifts` preset: 20 series, 2000 points each, with generated `feature_*` columns. The preview below shows the first provider with ground-truth change-point markers.

In [2]:
dataset = preset_dataset("extreme_mean_shifts", n_series=20, seed=42, series_length=2000)
print(f"Series count: {len(dataset)}")

provider = dataset[0]
print(f"Provider: {provider.name}")
print(f"Change points: {list(provider.change_points)}")

df = provider.dataset()
feature_columns = [column for column in df.columns if column.startswith("feature_")]
n_features = len(feature_columns)
vis = MultivariateTimeseriesVisualizer(
    backend=DrawBackend.MATPLOTLIB,
    dimensionality=n_features,
)
vis.set_data_provider(provider)
vis.set_plot_opts(axes=list(range(n_features)), title=provider.name)
for index, column in enumerate(feature_columns):
    vis.set_draw_opts(axes=[index], label=column)
    vis.set_plot_opts(axes=[index], ylabel=column)

fig, axes = plt.subplots(n_features, 1, figsize=(12, 2 * n_features), sharex=True)
axes = np.atleast_1d(axes).tolist()
ax_map = {f"timeseries_{index}": axes[index] for index in range(n_features)}
vis.draw(figure=fig, axes=ax_map)

cp_lines = (
    VerticalLineVisualComponent(DrawBackend.MATPLOTLIB)
    .set_lines(list(provider.change_points))
    .set_style(
        color="red",
        linestyle="solid",
        linewidth=1.5,
        alpha=0.8,
        label="Ground Truth",
        legend=True,
    )
)
for index, ax in enumerate(axes):
    cp_lines.draw(fig, ax, add_legend=(index == 0))
    handles, labels = ax.get_legend_handles_labels()
    if labels:
        ax.legend(loc="upper left", fontsize=8)

plt.tight_layout()
plt.show()
plt.close(fig)

## Algorithms

Five online CPD algorithms share the same learning period. Shewhart and Bayesian BOCPD use their standard defaults; Page, Crosier, and Variance CUSUM use ``delta=0.5``.


In [3]:
# Create two algorithms with different configs

LEARNING_PERIOD_SIZE = 100

algorithms = {
    "Shewhart": ShewhartControlChart(
        learning_period_size=LEARNING_PERIOD_SIZE,
        window_size=50,
    ),
    "Page CUSUM": PageTwoSidedCusum(
        learning_period_size=LEARNING_PERIOD_SIZE,
        delta=0.5,
    ),
    "Crosier CUSUM": CrosierCusum(
        learning_period_size=LEARNING_PERIOD_SIZE,
        delta=0.5,
    ),
    "Variance CUSUM": VarianceTwoSidedCUSUM(
        learning_period_size=LEARNING_PERIOD_SIZE,
        delta=0.5,
    ),
    "BOCPD": UnivariateGaussianConjugateBOCPD(
        learning_period_size=LEARNING_PERIOD_SIZE,
        hazard_lambda=10_000,
        prior_beta=1000,
    ),
}

for name, algo in algorithms.items():
    print(f"{name:20s} {algo.description}")

## Benchmark Entries and Report

Each algorithm is wrapped in an ``OnlineNoResetBenchmarkEntry`` with auto-scaled thresholds (50 points). The ``ColumnsSelectorTransformer`` extracts ``feature_0`` or ``feature_1`` from each provider. Classification semantics are set on ``OnlineNoResetBenchmark`` via ``max_delay=100`` and ``global_policy=NoResetPolicyKind.MIXED`` (non-strict threshold comparison).


In [4]:
# Create benchmark entries directly for each algorithm
feature_transformer_0 = ColumnsSelectorTransformer(columns=["feature_0"])
feature_transformer_1 = ColumnsSelectorTransformer(columns=["feature_1"])

# Since latest API changes, bisegment_cut is configured per benchmark entry.
entry_bisegment_cut = BisegmentCut.parse((100, 0))

entries = {
    "Shewhart [feature 0]": OnlineNoResetBenchmarkEntry(
        algorithm=algorithms["Shewhart"],
        thresholds=AutoThresholdsRange(count=50),
        data_transformer=feature_transformer_0,
        bisegment_cut=entry_bisegment_cut,
    ),
    "Shewhart [feature 1]": OnlineNoResetBenchmarkEntry(
        algorithm=algorithms["Shewhart"],
        thresholds=AutoThresholdsRange(count=50),
        data_transformer=feature_transformer_1,
        bisegment_cut=entry_bisegment_cut,
    ),
    "Page CUSUM": OnlineNoResetBenchmarkEntry(
        algorithm=algorithms["Page CUSUM"],
        thresholds=AutoThresholdsRange(count=50),
        data_transformer=feature_transformer_0,
        bisegment_cut=entry_bisegment_cut,
    ),
    "Crosier CUSUM": OnlineNoResetBenchmarkEntry(
        algorithm=algorithms["Crosier CUSUM"],
        thresholds=AutoThresholdsRange(count=50),
        data_transformer=None,
        bisegment_cut=entry_bisegment_cut,
    ),
    "Variance CUSUM": OnlineNoResetBenchmarkEntry(
        algorithm=algorithms["Variance CUSUM"],
        thresholds=AutoThresholdsRange(count=50),
        data_transformer=feature_transformer_0,
        bisegment_cut=entry_bisegment_cut,
    ),
    "BOCPD": OnlineNoResetBenchmarkEntry(
        algorithm=algorithms["BOCPD"],
        thresholds=AutoThresholdsRange(count=50),
        data_transformer=feature_transformer_0,
        bisegment_cut=entry_bisegment_cut,
    ),
}

In [5]:
benchmark = OnlineNoResetBenchmark(
    dataset=dataset,
    registry=BenchmarkRegistry(),
    max_delay=10,
    global_policy=NoResetPolicyKind.MIXED,
    policy_strict=False,
)

## Global and Transition Results

Running all entries in a single `get_classification_table()` call creates cached no-reset bisegment traces. The global view below shows only PR-AUC. For one selected transition, we show both PR-AUC and delay-vs-ARL to connect classification quality with false-alarm robustness.

In [6]:
global_tables = benchmark.get_classification_table(list(entries.values()))
global_pr_auc = benchmark.get_pr_auc_table(global_tables)

selected_transition = next(iter(dataset.transitions))
transition_tables = benchmark.get_classification_table_by_transition(
    list(entries.values()),
    transition=selected_transition,
    use_arl=True,
    arl_length=1000,
)
transition_pr_auc = benchmark.get_pr_auc_table(transition_tables)

print("Selected transition:", selected_transition)
for name, entry in entries.items():
    print(
        f"{name:20s} global PR-AUC = {global_pr_auc[entry.description]:.6f}; "
        f"transition PR-AUC = {transition_pr_auc[entry.description]:.6f}"
    )

In [7]:
entry_colors = {
    "Shewhart [feature 0]": "tab:blue",
    "Shewhart [feature 1]": "tab:orange",
    "Page CUSUM": "tab:green",
    "Crosier CUSUM": "tab:red",
    "Variance CUSUM": "tab:purple",
    "BOCPD": "tab:brown",
}

global_table_dict = {name: global_tables[entry.description] for name, entry in entries.items()}
global_plotter = (
    BenchmarkPlotter()
    .set_benchmark_tables(global_table_dict)
    .set_metrics({"global_pr_auc": PrAucVisualizer(label="Global PR-AUC")})
)
for name, color in entry_colors.items():
    global_plotter.set_entry_draw_opts(entry=name, color=color)

fig, ax = plt.subplots(1, 1, figsize=(7, 5))
global_plotter.draw(figure=fig, axes={"global_pr_auc": ax})
ax.set_title("Global PR-AUC")
plt.tight_layout()
plt.show()
plt.close(fig)

In [8]:
transition_table_dict = {name: transition_tables[entry.description] for name, entry in entries.items()}
transition_plotter = (
    BenchmarkPlotter()
    .set_benchmark_tables(transition_table_dict)
    .set_metrics(
        {
            "transition_pr_auc": PrAucVisualizer(label="Transition PR-AUC"),
            "delay_vs_arl": ARLBasedMetricVisualizer(
                y_metrics=["median_delay"],
                title="Transition median delay vs ARL",
                ylabel="Median delay (steps)",
            ),
        }
    )
)
for name, color in entry_colors.items():
    transition_plotter.set_entry_draw_opts(entry=name, color=color)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
transition_plotter.draw(
    figure=fig,
    axes={"transition_pr_auc": axes[0], "delay_vs_arl": axes[1]},
)
fig.suptitle(f"Transition summary: {selected_transition}", fontsize=14)
plt.tight_layout()
plt.show()
plt.close(fig)

## Focus: Variance CUSUM

Variance CUSUM detects changes in scale, not location. On mean-shift data it is expected to underperform. The detailed plots below show exactly where it struggles.


In [9]:
# USE BENCHMARKING VISUALIZATION
var_entry = entries["Variance CUSUM"]
var_table = global_tables[var_entry.description].sort_values("threshold").reset_index(drop=True)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

ax1.plot(var_table["threshold"], var_table["f1"], color="tab:red")
ax1.set_xlabel("Threshold")
ax1.set_ylabel("F1")
ax1.set_title("F1 vs Threshold")
ax1.grid(alpha=0.3)

ax2.plot(var_table["threshold"], var_table["precision"], label="Precision")
ax2.plot(var_table["threshold"], var_table["recall"], label="Recall")
ax2.set_xlabel("Threshold")
ax2.set_ylabel("Score")
ax2.set_title("Precision / Recall vs Threshold")
ax2.legend()
ax2.grid(alpha=0.3)

ax3.plot(var_table["threshold"], var_table["mean_delay"], label="Mean")
ax3.plot(var_table["threshold"], var_table["median_delay"], label="Median")
ax3.set_xlabel("Threshold")
ax3.set_ylabel("Delay")
ax3.set_title("Delay vs Threshold")
ax3.legend()
ax3.grid(alpha=0.3)

plt.tight_layout()
plt.show()
plt.close(fig)

## Bisegment Analysis

At the chosen threshold below, which individual bisegments does Variance CUSUM handle correctly, and which does it miss? The per-bisegment classification table answers this.

In [10]:
chosen_threshold = 10.0
bs_tables = benchmark.get_bisegments_table([var_entry], threshold=chosen_threshold)
bs_table = next(iter(bs_tables.values())).copy().reset_index(drop=True)
display(bs_table)

## Trace for One Bisegment

Pick one bisegment from the table and inspect the cached benchmark trace. The no-reset trace itself does not store declared detections; the green vertical lines below mark threshold-crossing candidates for the chosen threshold.

In [11]:
row = bs_table.iloc[min(39, len(bs_table) - 1)]
target_name = row["bisegment_name"]
registry_items = list(benchmark.registry.executions_registry.items())

matching_items = [
    (descr, single_run)
    for descr, single_run in registry_items
    if descr.detector_description == var_entry.description and descr.provider_description.name == target_name
]
if not matching_items:
    matching_items = [
        (descr, single_run) for descr, single_run in registry_items if descr.provider_description.name == target_name
    ]
if not matching_items:
    available_names = sorted({descr.provider_description.name for descr, _ in registry_items})
    raise LookupError(
        f"No cached run found for bisegment '{target_name}'. Available names count: {len(available_names)}"
    )

key, run = matching_items[0]
provider = run.provider
trace = run.trace
transformed = var_entry.data_transformer.transform(provider) if var_entry.data_transformer is not None else provider
threshold_crossings = np.flatnonzero(
    np.asarray(trace.detection_function, dtype=np.float64) >= chosen_threshold
).tolist()
threshold_crossings = [
    point
    for index, point in enumerate(threshold_crossings)
    if index == 0 or threshold_crossings[index - 1] != point - 1
]

plotter = OnlineCpdPlotter(
    backend=DrawBackend.MATPLOTLIB,
    data_provider=transformed,
    detection_trace=trace,
)
plotter.set_ground_truth(list(provider.change_points))
plotter.set_legend_axis("detection_function")
plotter.remove_component("skip_fill")
plotter.timeseries_visualizer.set_plot_opts(title=f"Bisegment: {provider.annotation.name}")
plotter.trace_visualizer.set_detection_func_plot_opts(title="Detection trace")

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
fig = plotter.draw(
    figure=fig,
    axes={"timeseries": axes[0], "detection_function": axes[1]},
)
axes[1].axhline(chosen_threshold, color="green", linestyle="--", linewidth=2, label="Chosen Threshold")
for index, point in enumerate(threshold_crossings):
    axes[0].axvline(point, color="green", linestyle="--", linewidth=2, alpha=0.8)
    axes[1].axvline(
        point,
        color="green",
        linestyle="--",
        linewidth=2,
        alpha=0.8,
        label="Threshold Crossing" if index == 0 else None,
    )
for ax in axes:
    handles, labels = ax.get_legend_handles_labels()
    if labels:
        ax.legend(loc="upper left", fontsize=9)
plt.tight_layout()
plt.show()
plt.close(fig)
print("Registry key:", key)